In [4]:
import os

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

print("Data Lake folders created successfully!")


Data Lake folders created successfully!


In [10]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, to_date

spark = SparkSession.builder \
    .appName("Data_Lake_Assignment") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

df_bronze = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)

print("=== Original Schema ===")
df_bronze.printSchema()

total_records = df_bronze.count()
print(f"Total Original Records: {total_records}")

print("\n=== First 5 Rows ===")
df_bronze.show(5)

print("\n=== Identifying Data Issues ===")

duplicate_count = total_records - df_bronze.dropDuplicates().count()
print(f"Exact Duplicate Rows: {duplicate_count}")

null_counts = df_bronze.select([count(when(col(c).isNull() | (col(c).cast("string") == ""), c)).alias(c) for c in df_bronze.columns])
print("\nNull or Empty Values per column:")
null_counts.show()

invalid_quantities = df_bronze.filter(col("quantity") < 0).count()
invalid_prices = df_bronze.filter(col("unit_price") <= 0).count()
invalid_discounts = df_bronze.filter((col("discount_percent") < 0) | (col("discount_percent") > 100)).count()

print(f"Rows with negative quantity: {invalid_quantities}")
print(f"Rows with zero or negative unit price: {invalid_prices}")
print(f"Rows with invalid discount (not 0-100): {invalid_discounts}")

invalid_dates = df_bronze.withColumn("parsed_date", to_date(col("order_date"), "yyyy-MM-dd")) \
                         .filter(col("parsed_date").isNull() & col("order_date").isNotNull()).count()
print(f"Rows with inconsistent/invalid dates: {invalid_dates}")


=== Original Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Total Original Records: 1000

=== First 5 Rows ===
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+----------

In [12]:
from pyspark.sql.functions import initcap, trim, abs

df_silver = df_bronze.dropDuplicates()

df_silver = df_silver.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "state": "Unknown"
})

for c in ["customer_name", "product", "category", "payment_method", "city", "state"]:
    df_silver = df_silver.withColumn(c, initcap(trim(col(c))))

df_silver = df_silver.withColumn("quantity", when(col("quantity") < 0, abs(col("quantity"))).otherwise(col("quantity")))
df_silver = df_silver.withColumn("unit_price", when(col("unit_price") <= 0, None).otherwise(col("unit_price")))
df_silver = df_silver.withColumn("discount_percent", when((col("discount_percent") < 0) | (col("discount_percent") > 100), 0).otherwise(col("discount_percent")))

df_silver = df_silver.dropna(subset=["unit_price"])

df_silver = df_silver.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
df_silver = df_silver.dropna(subset=["order_date"])

df_silver.write.mode("overwrite").csv("data_lake/silver/preprocessed_sales", header=True)


In [13]:
df_silver_read = spark.read.csv("data_lake/silver/preprocessed_sales", header=True, inferSchema=True)

print(f"Row count BEFORE preprocessing (Bronze): {total_records}")
print(f"Row count AFTER preprocessing (Silver): {df_silver_read.count()}")

print("\n=== Cleaned Data (First 5 Rows) ===")
df_silver_read.show(5)


Row count BEFORE preprocessing (Bronze): 1000
Row count AFTER preprocessing (Silver): 976

=== Cleaned Data (First 5 Rows) ===
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|       670|              30|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|       670|               0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  

### Preprocessing Operations Performed:
1. **Duplicates:** Removed all exact duplicate rows.
2. **Missing Values:** Replaced null values in text columns (customer_name, payment_method, state) with "Unknown".
3. **Text Formatting:** Trimmed extra spaces and applied Title Case (`initcap`) to all text fields for consistency.
4. **Numerical Validation:**
   - Converted negative quantities to absolute (positive) numbers.
   - Nullified and dropped records where unit_price was 0 or negative.
   - Reset invalid discount_percent values (below 0 or above 100) to 0.
5. **Date Formatting:** Converted `order_date` to standard `yyyy-MM-dd` format and dropped rows with unparseable/invalid dates.
